# noisy: шесть ракурсов отдельно

В предыдущем опыте усредняли DINOv2 embeddings шести ракурсов и брали максимум. Здесь сохраняем порядок ракурсов и передаём все шесть в CatBoost. Используем те же train и validation.

## 1. Данные и сохранённые embeddings

In [ ]:
%pip install -q catboost iterative-stratification

from pathlib import Path
import hashlib
import json
import zipfile

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data/train.csv').is_file()), None)
if ROOT is None:
    ROOT = Path('/content/drive/MyDrive/Colab Notebooks/colab_upload/AI challenge main')
package = ROOT / 'noisy_dinov2_assets.zip'
if package.is_file():
    with zipfile.ZipFile(package) as archive:
        for name in archive.namelist():
            if not (ROOT / name).is_file():
                archive.extract(name, ROOT)
assert (ROOT / 'data/train.csv').is_file(), ROOT

labels = pd.read_csv(ROOT / 'data/train.csv')
targets = labels.columns.drop('item_id').tolist()
split = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=.15, random_state=42)
train_val_idx, test_idx = next(split.split(labels, labels[targets]))
train_val = labels.iloc[train_val_idx].reset_index(drop=True)
split = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=.1765, random_state=43)
train_idx, val_idx = next(split.split(train_val, train_val[targets]))
train = train_val.iloc[train_idx].reset_index(drop=True)
val = train_val.iloc[val_idx].reset_index(drop=True)
objects = pd.concat([train, val], ignore_index=True)
assert (len(train), len(val), len(test_idx)) == (6259, 1361, 1344)
assert set(train.item_id).isdisjoint(val.item_id)

cache = ROOT / 'labels/simple/models/dinov2'
if not (cache / 'embeddings.npy').is_file():
    cache = ROOT / 'labels/noisy/artifacts'
    embedding_file = cache / 'dinov2_embeddings.npy'
else:
    embedding_file = cache / 'embeddings.npy'

manifest = json.loads((cache / 'embeddings_manifest.json').read_text(encoding='utf-8'))
assert manifest['model'] == 'dinov2_vits14'
assert manifest['item_ids'] == objects.item_id.tolist()
assert manifest['labels_sha256'] == hashlib.sha256((ROOT / 'data/train.csv').read_bytes()).hexdigest()
assert np.load(cache / 'embeddings_done.npy').all()
views = np.load(embedding_file, mmap_mode='r')
assert views.shape == (7620, 6, 384) and np.isfinite(views).all()
visual = views.reshape(len(views), -1)
print('Train:', len(train), 'Validation:', len(val), '| признаки:', visual.shape)

## 2. Та же геометрия

In [ ]:
config = json.loads((ROOT / 'archive/artifacts/catboost_features_config.json').read_text(encoding='utf-8'))
base_columns = config['feature_columns']
local_columns = ['pca_curvature_mean', 'pca_curvature_p90', 'pca_curvature_p95',
                 'normal_variation_p50', 'normal_variation_p90', 'normal_variation_p95',
                 'normal_variation_std', 'dihedral_over_10_share', 'dihedral_over_30_share']
base = pd.read_csv(ROOT / 'archive/artifacts/train_mesh_features_relative.csv').set_index('item_id')
local = pd.read_csv(ROOT / 'labels/lowpoly/artifacts/geometry_features.csv').set_index('item_id')
smooth = pd.read_csv(ROOT / 'labels/noisy/artifacts/smoothing_features.csv').set_index('item_id')
smooth_columns = [c for c in smooth if c.startswith('smooth_')]
assert len(base_columns) == 44 and len(local_columns) == 9 and len(smooth_columns) == 14
assert base.index.is_unique and local.index.is_unique and smooth.index.is_unique
geometry = base.join(local[local_columns]).join(smooth[smooth_columns])
geometry = geometry.loc[objects.item_id, base_columns + local_columns + smooth_columns].to_numpy(dtype=np.float32)
assert geometry.shape == (7620, 67) and not np.isinf(geometry).any()
print('Геометрия:', geometry.shape)

## 3. CatBoost

In [ ]:
out = ROOT / 'labels/noisy/models/six_views'
out.mkdir(parents=True, exist_ok=True)
y_train = train.noisy.to_numpy()
y_val = val.noisy.to_numpy()
n_train = len(train)
grid = np.arange(.05, .96, .01)

def score(p, threshold):
    prediction = p >= threshold
    tn, fp, fn, tp = confusion_matrix(y_val, prediction, labels=[0, 1]).ravel()
    return {'Precision': precision_score(y_val, prediction, zero_division=0),
            'Recall': recall_score(y_val, prediction, zero_division=0),
            'F1': f1_score(y_val, prediction, zero_division=0),
            'AP': average_precision_score(y_val, p),
            'FP': int(fp), 'FN': int(fn), 'Threshold': float(threshold)}

results = []
probabilities = val[['item_id', 'noisy']].copy()
for name, X in {
    'six_views': visual,
    'six_views_geometry': np.concatenate([visual, geometry], axis=1),
}.items():
    model = CatBoostClassifier(**config['parameters'])
    model.fit(X[:n_train], y_train, eval_set=(X[n_train:], y_val),
              early_stopping_rounds=50, verbose=False)
    p = model.predict_proba(X[n_train:])[:, 1]
    threshold = float(max(grid, key=lambda t: f1_score(y_val, p >= t)))
    results.append({'Model': name, **score(p, threshold)})
    probabilities[name] = p
    model.save_model(str(out / f'{name}.cbm'))
    print(name, '| F1:', round(results[-1]['F1'], 4), '| AP:', round(results[-1]['AP'], 4), flush=True)

table = pd.DataFrame(results)
display(table.round(4))
table.to_csv(out / 'metrics.csv', index=False)
probabilities.to_csv(out / 'validation_probabilities.csv', index=False)

## 4. Сравнение

In [ ]:
previous = pd.DataFrame([
    {'Model': 'DINOv2 mean + max', 'F1': .8065, 'AP': .8776},
    {'Model': 'DINOv2 mean + max + geometry', 'F1': .8301, 'AP': .8903},
])
comparison = pd.concat([previous, table[['Model', 'F1', 'AP']]], ignore_index=True)
display(comparison.round(4))
print('Предыдущие числа взяты из выполненного noisy_dinov2.ipynb.')
print('Новые модели сравнивались только на validation. Internal test не использовали.')

## Вывод

Отдельные ракурсы не улучшили результат: F1 = 0.8032 без геометрии и 0.8237 с ней. У прежнего mean + max + геометрия F1 = 0.8301 на той же validation. Поэтому оставляем прежний вариант. Порог и модель подбирались на validation, поэтому оценки могут быть немного оптимистичными.